# Notebook 2: LSTM with Attention
Run Notebook 1 first to generate outputs/ files.

Architecture: CNN → LSTM → Attention → Dense → Sigmoid
Inspired by Benchaji et al. (2021) and Akour et al. (2025)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import warnings, joblib, os
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Preprocessed Data

In [ ]:
X_train = np.load('outputs/X_train_sm.npy').astype(np.float32)
y_train = np.load('outputs/y_train_sm.npy').astype(np.float32)
X_val   = np.load('outputs/X_val_sc.npy').astype(np.float32)
y_val   = np.load('outputs/y_val.npy').astype(np.float32)
X_test  = np.load('outputs/X_test_sc.npy').astype(np.float32)
y_test  = np.load('outputs/y_test.npy').astype(np.float32)

N_FEATURES = X_train.shape[1]
SEQ_LEN    = 10   # Window size: treat 10 consecutive features as a sequence

print(f'Train: {X_train.shape} | Fraud: {int(y_train.sum())}')
print(f'Val:   {X_val.shape}   | Fraud: {int(y_val.sum())}')
print(f'Test:  {X_test.shape}  | Fraud: {int(y_test.sum())}')
print(f'N_FEATURES: {N_FEATURES}')

## 2. Dataset — Sliding Window Sequences

Since PaySim accounts are mostly unique, we create sequences by treating
each transaction's feature vector as a single timestep and building
sliding windows over the time-sorted transaction stream.
This captures temporal burst patterns (the Velocity Agent rationale).

In [ ]:
class FraudSequenceDataset(Dataset):
    """
    Creates sliding windows of SEQ_LEN transactions.
    Label = fraud label of the LAST transaction in the window.
    This mimics a real-time detector: given the last N transactions
    in the stream, predict if the current one is fraud.
    """
    def __init__(self, X, y, seq_len=10):
        self.X       = torch.FloatTensor(X)
        self.y       = torch.FloatTensor(y)
        self.seq_len = seq_len
        # Only valid from index seq_len-1 onwards
        self.valid_idx = np.arange(seq_len - 1, len(X))

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, idx):
        end   = self.valid_idx[idx] + 1
        start = end - self.seq_len
        seq   = self.X[start:end]          # (seq_len, n_features)
        label = self.y[self.valid_idx[idx]] # scalar
        return seq, label

BATCH_SIZE = 2048  # Large batch fits well on RTX 5060

train_ds = FraudSequenceDataset(X_train, y_train, SEQ_LEN)
val_ds   = FraudSequenceDataset(X_val,   y_val,   SEQ_LEN)
test_ds  = FraudSequenceDataset(X_test,  y_test,  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 3. Model Architecture: CNN → LSTM → Attention → Dense

In [ ]:
class AttentionLayer(nn.Module):
    """Bahdanau-style additive attention over LSTM hidden states."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        scores  = self.v(torch.tanh(self.W(lstm_out)))  # (batch, seq_len, 1)
        weights = F.softmax(scores, dim=1)              # (batch, seq_len, 1)
        context = (weights * lstm_out).sum(dim=1)       # (batch, hidden_dim)
        return context, weights.squeeze(-1)


class CNNLSTMAttention(nn.Module):
    """
    Hybrid CNN-LSTM-Attention model for fraud detection.
    Follows architecture from Akour et al. (2025) adapted for PaySim.

    Flow:
      Input (batch, seq_len, features)
      → CNN extracts local spatial patterns across features
      → LSTM captures temporal dependencies across the sequence
      → Attention highlights most fraud-relevant timesteps
      → Dense → Sigmoid → fraud probability
    """
    def __init__(self, n_features, seq_len, cnn_filters=64, cnn_kernel=3,
                 lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        # CNN: operates on (batch, features, seq_len) — treats features as channels
        self.conv1 = nn.Conv1d(n_features, cnn_filters, kernel_size=cnn_kernel, padding=1)
        self.conv2 = nn.Conv1d(cnn_filters, cnn_filters, kernel_size=cnn_kernel, padding=1)
        self.pool  = nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
        self.bn1   = nn.BatchNorm1d(cnn_filters)

        # LSTM: input is (batch, seq_len, cnn_filters)
        self.lstm  = nn.LSTM(
            input_size=cnn_filters,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
            bidirectional=False
        )

        # Attention
        self.attention = AttentionLayer(lstm_hidden)

        # Classifier head
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(lstm_hidden, 64)
        self.bn2     = nn.BatchNorm1d(64)
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        # CNN expects (batch, channels, length) → transpose
        x_cnn = x.permute(0, 2, 1)                      # (batch, n_features, seq_len)
        x_cnn = F.relu(self.conv1(x_cnn))
        x_cnn = F.relu(self.conv2(x_cnn))
        x_cnn = self.pool(x_cnn)
        x_cnn = self.bn1(x_cnn)
        x_cnn = x_cnn.permute(0, 2, 1)                  # (batch, seq_len', cnn_filters)

        # LSTM
        lstm_out, _ = self.lstm(x_cnn)                  # (batch, seq_len', lstm_hidden)

        # Attention
        context, attn_weights = self.attention(lstm_out) # (batch, lstm_hidden)

        # Classifier
        out = self.dropout(context)
        out = F.relu(self.bn2(self.fc1(out)))
        out = self.dropout(out)
        out = torch.sigmoid(self.fc2(out)).squeeze(-1)   # (batch,)
        return out, attn_weights


model = CNNLSTMAttention(
    n_features=N_FEATURES,
    seq_len=SEQ_LEN,
    cnn_filters=64,
    cnn_kernel=3,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 4. Training Setup

In [ ]:
# Weighted loss: penalise missing fraud more than false alarms
# fraud_weight = ratio of non-fraud to fraud in ORIGINAL (pre-SMOTE) train
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Fraud weight for loss: {fraud_weight:.1f}x')

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5, verbose=True
)

EPOCHS    = 30
best_val_f1 = 0
train_losses, val_losses = [], []
train_f1s,    val_f1s    = [], []

## 5. Training Loop

In [ ]:
def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_preds, all_labels = [], []

    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            if train:
                optimizer.zero_grad()

            preds, _ = model(X_batch)
            loss      = criterion(preds, y_batch)

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(y_batch)
            all_preds.append(preds.cpu().detach().numpy())
            all_labels.append(y_batch.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    avg_loss   = total_loss / len(loader.dataset)
    f1         = f1_score(all_labels, (all_preds >= 0.5).astype(int), zero_division=0)
    return avg_loss, f1, all_preds, all_labels


print(f'Training for {EPOCHS} epochs on {DEVICE}...')
print(f'{"Epoch":>5} {"Train Loss":>12} {"Train F1":>10} {"Val Loss":>10} {"Val F1":>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, _, _ = run_epoch(train_loader, train=True)
    vl_loss, vl_f1, _, _ = run_epoch(val_loader,   train=False)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_f1s.append(tr_f1)
    val_f1s.append(vl_f1)

    scheduler.step(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), 'outputs/best_lstm_model.pt')

    print(f'{epoch:>5} {tr_loss:>12.5f} {tr_f1:>10.4f} {vl_loss:>10.5f} {vl_f1:>8.4f}')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss', color='#2196F3')
ax1.plot(val_losses,   label='Val Loss',   color='#F44336')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.legend()

ax2.plot(train_f1s, label='Train F1', color='#2196F3')
ax2.plot(val_f1s,   label='Val F1',   color='#F44336')
ax2.set_title('Training & Validation F1 Score')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.legend()

plt.tight_layout()
plt.savefig('outputs/lstm_training_curves.png', dpi=150)
plt.show()
print('Saved: outputs/lstm_training_curves.png')

## 7. Test Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('outputs/best_lstm_model.pt', map_location=DEVICE))

_, _, test_preds, test_labels = run_epoch(test_loader, train=False)

# Find optimal threshold via val set
_, _, val_preds, val_labels = run_epoch(val_loader, train=False)
thresholds = np.arange(0.1, 0.9, 0.05)
best_thresh, best_f1 = 0.5, 0
for t in thresholds:
    f1 = f1_score(val_labels, (val_preds >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t
print(f'Optimal threshold from val set: {best_thresh:.2f} (F1={best_f1:.4f})')

y_pred_final = (test_preds >= best_thresh).astype(int)

lstm_results = {
    'Model':     'CNN-LSTM-Attention',
    'Precision': round(precision_score(test_labels, y_pred_final, zero_division=0), 4),
    'Recall':    round(recall_score(test_labels, y_pred_final), 4),
    'F1':        round(f1_score(test_labels, y_pred_final), 4),
    'ROC-AUC':   round(roc_auc_score(test_labels, test_preds), 4),
    'PR-AUC':    round(average_precision_score(test_labels, test_preds), 4),
}

print('\n=== CNN-LSTM-ATTENTION TEST RESULTS ===')
for k, v in lstm_results.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print(f'\n{classification_report(test_labels, y_pred_final, target_names=["Legitimate","Fraud"])}')

# Save probabilities for synthesis notebook
np.save('outputs/lstm_test_proba.npy', test_preds)
import json
with open('outputs/lstm_results.json', 'w') as f:
    json.dump(lstm_results, f, indent=2)
print('Saved lstm results')

In [ ]:
# Load XGBoost results for comparison
try:
    xgb_results = joblib.load('outputs/xgboost_gpu.pkl')  # or just use predictions if saved
    # Better: load from all_results.csv later in synthesis
    print("XGBoost model available for ensemble/comparison")
except:
    print("XGBoost not found (run Notebook 1 first)")

## 8. Confusion Matrix + Precision-Recall Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(test_labels, y_pred_final)
sns_data = pd.DataFrame(cm,
    index=['Actual: Legit', 'Actual: Fraud'],
    columns=['Pred: Legit', 'Pred: Fraud']
)
import seaborn as sns
sns.heatmap(sns_data, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('CNN-LSTM-Attention Confusion Matrix')

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(test_labels, test_preds)
pr_auc = average_precision_score(test_labels, test_preds)
ax2.plot(rec, prec, color='#2196F3', lw=2, label=f'CNN-LSTM-Attn (PR-AUC={pr_auc:.4f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/lstm_evaluation.png', dpi=150)
plt.show()
print('Saved: outputs/lstm_evaluation.png')